In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class MToE(nn.Module):
    """
    A minimal, faithful PyTorch implementation of the MToE block described in your screenshot.

    Notation / shapes (with batch):
      m = #modalities
      l = #tokens per modality
      M = m*l = total tokens
      d = feature dim
      n = #experts
      p = #tasks (task slots)

    Inputs:
      X_list: list length m, each tensor [B, l, d]  (token features per modality)

    Learnables:
      Phi: [d, n, p]  (routing parameters)
      TE:  [p, d]     (task embeddings shared across experts)

    Internals:
      X = concat(X_list) -> [B, M, d]
      R = XΦ -> [B, M, n, p]  (routing scores)

      Dispatch weights D: softmax over tokens (dim=1) for each (expert j, task k)
        D: [B, M, n, p]
        X_tilde[j,k] = sum_i D[i,j,k] * X[i] -> [B, n, p, d]

      Expert outputs:
        Y_tilde[j,k] = f_j(X_tilde[j,k] + TE[k]) -> [B, n, p, d]

      Combine weights C: softmax over (expert,task) per token i
        C: [B, M, n, p]
        w[j,k] = sum_i C[i,j,k]  -> [B, n, p] (then normalized over experts per task)
        Y[k] = sum_j w[j,k] * Y_tilde[j,k] -> [B, p, d]
    """

    def __init__(self, d: int, n: int, p: int, experts: list[nn.Module], eps: float = 1e-9):
        super().__init__()
        assert len(experts) == n, "Need exactly n expert modules."
        self.d, self.n, self.p = d, n, p
        self.experts = nn.ModuleList(experts)

        # Routing parameter tensor Φ in the paper: R^{d × n × p}
        self.Phi = nn.Parameter(torch.randn(d, n, p) * (d ** -0.5))

        # Task embedding TE_k shared across experts: R^{p × d}
        self.TE = nn.Parameter(torch.randn(p, d) * (d ** -0.5))

        self.eps = eps

    def forward(self, X_list: list[torch.Tensor], return_router_tensors: bool = False):
        """
        Returns:
          Y: [B, p, d]  (one output slot per task)
          optionally (R, D, C) for inspection / probability modeling
        """
        # ---- 1) concatenate modality tokens ----
        # X_list: m * [B, l, d]
        B = X_list[0].shape[0]
        d = X_list[0].shape[-1]
        assert d == self.d
        X = torch.cat(X_list, dim=1)  # [B, M, d]
        _, M, _ = X.shape

        # ---- 2) routing scores R = XΦ ----
        # R[b, i, j, k] = <X[b,i,:], Phi[:,j,k]>
        R = torch.einsum("bmd,dnp->bmnp", X, self.Phi)  # [B, M, n, p]

        # ---- 3) dispatch weights D: softmax over tokens for each (j,k) ----
        # D[:, :, j, k] sums to 1 over token dimension
        D = F.softmax(R, dim=1)  # [B, M, n, p]

        # ---- 4) build expert/task slots by weighted sum of tokens ----
        # X_tilde[b, j, k, :] = sum_i D[b,i,j,k] * X[b,i,:]
        X_tilde = torch.einsum("bmnp,bmd->bnpd", D, X)  # [B, n, p, d]

        # ---- 5) expert processing per expert j, for all task slots k ----
        # Add shared task embedding TE[k] to every expert's kth slot
        X_in = X_tilde + self.TE.view(1, 1, self.p, self.d)  # [B, n, p, d]

        Y_tilde = torch.empty_like(X_in)  # [B, n, p, d]
        for j, f_j in enumerate(self.experts):
            # Process all p slots for expert j in one go: [B*p, d] -> [B*p, d]
            inp = X_in[:, j, :, :].reshape(B * self.p, self.d)
            out = f_j(inp).reshape(B, self.p, self.d)
            Y_tilde[:, j, :, :] = out

        # ---- 6) combine weights C: softmax over (j,k) per token i ----
        # For each token i, distribute probability mass across all expert-task slots.
        C_flat = F.softmax(R.reshape(B, M, self.n * self.p), dim=2)  # [B, M, n*p]
        C = C_flat.reshape(B, M, self.n, self.p)  # [B, M, n, p]

        # Aggregate token-wise combine mass into per-(j,k) weights
        w = C.sum(dim=1)  # [B, n, p]

        # Normalize across experts per task to make it a convex combination over experts for each task
        w = w / (w.sum(dim=1, keepdim=True) + self.eps)  # [B, n, p]

        # Final task outputs: Y[b,k,:] = sum_j w[b,j,k] * Y_tilde[b,j,k,:]
        Y = torch.einsum("bnp,bnpd->bpd", w, Y_tilde)  # [B, p, d]

        if return_router_tensors:
            # R is the thing they later reinterpret probabilistically.
            return Y, (R, D, C, w)
        return Y


# ---- Example expert definition ----
def make_mlp_expert(d: int, hidden: int = 4):
    return nn.Sequential(
        nn.Linear(d, hidden * d),
        nn.GELU(),
        nn.Linear(hidden * d, d),
    )


In [ ]:
# ---- Quick usage ----
B, m, l, d = 2, 3, 4, 16
n, p = 5, 7

experts = [make_mlp_expert(d) for _ in range(n)]
mtoe = MToE(d=d, n=n, p=p, experts=experts)

X_list = [torch.randn(B, l, d) for _ in range(m)]
Y, (R, D, C, w) = mtoe(X_list, return_router_tensors=True)

print("X_list:", [x.shape for x in X_list])
print("R:", R.shape, "D:", D.shape, "C:", C.shape, "w:", w.shape, "Y:", Y.shape)

In [ ]:
import torch 

